# RootCause-SLM: pipeline and benchmark

Runs `src/pipeline.py` (detection + explanation) and `src/benchmark.py` on a Colab GPU. Set the runtime to T4 first (Runtime > Change runtime type).

In [ ]:
!git clone -q https://github.com/NafiyTP/RootCause-SLM.git
%cd /content/RootCause-SLM

In [ ]:
!pip install -q "transformers==4.46.3" peft accelerate
!pip install -q -U "torchao>=0.16.0"

If an import fails further down, restart the session (Runtime > Restart session) and run again from the next cell.

In [ ]:
%cd /content/RootCause-SLM
import torch, sklearn
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("scikit-learn", sklearn.__version__)

## Benchmark

Speed and cost on the T4 (64 test lines, batch sizes 1 and 16, a few minutes).

In [ ]:
!python src/benchmark.py

In [ ]:
from IPython.display import Markdown
Markdown(open('results/benchmark/benchmark.md', encoding='utf-8').read())

## Pipeline

Upload a raw HDFS log file with complete blocks: `raw_data/HDFS_window_3k.log` (3,000 consecutive blocks from the test period, 15 anomalies). A file of isolated lines like `HDFS_test_500.log` makes every block look cut, so everything gets flagged.

In [ ]:
from google.colab import files
up = files.upload()
LOG = list(up)[0]
print(LOG)

In [ ]:
!python src/pipeline.py --log "$LOG" --top 10

## Live demo

Starts a small web app with the detector and the real fine-tuned model. Open the `gradio.live` link it prints. The cell keeps running while the app is up; stop it to continue.

In [ ]:
!pip install -q gradio "huggingface-hub<1.0"
!python src/app.py --share

## Download the results

In [ ]:
!zip -qr results_colab.zip results/benchmark results/pipeline
from google.colab import files
files.download('results_colab.zip')